# 🕸️ 03 - Echo-Chamber Graph & Network Centrality Mapping

### Uncovering Information Contagion & Amplification Loops

Sensational financial stories do not exist in a vacuum; they circulate in **echo chambers** where multiple outlets parrot identical buzzwords, anonymous sources, and tickers without independent verification.

In this notebook, we model cross-outlet media dynamics as an **attributed undirected network**:
- **Nodes**: Media outlets (attributed with average hype, category, and flagged ratio).
- **Edges**: Weighted narrative alignment combining Named Entity Jaccard similarity and top article lexical convergence.
- **Centrality**: PageRank and Degree Centrality identify the primary **echo epicenters**.
- **Modularity**: Greedy modularity community detection partitions the graph into distinct thematic echo chambers.

---

In [1]:
# Ensure project root is in sys.path and is current working directory
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f'[OK] Working directory set to project root: {PROJECT_ROOT}')

import pandas as pd
import networkx as nx
from src.config import PROCESSED_DATA_DIR, FIGURES_DIR
from src.graph import EchoChamberGraphBuilder
from src.viz import plot_echo_chamber_network_plotly, export_pyvis_network_html

# Load enriched features
features_path = PROCESSED_DATA_DIR / 'sample_features.parquet'
if features_path.exists():
    df = pd.read_parquet(features_path)
else:
    df = pd.read_csv(PROCESSED_DATA_DIR / 'sample_features.csv')

# Build Echo-Chamber Graph
builder = EchoChamberGraphBuilder()
G = builder.build_outlet_graph(df, edge_threshold=0.15)

print(f'Graph Nodes (Outlets): {len(G.nodes)}')
print(f'Graph Edges (Echo Links): {len(G.edges)}')

summary_data = []
for node, data in G.nodes(data=True):
    summary_data.append({
        'Outlet': node,
        'Category': data.get('category'),
        'Avg Hype': data.get('avg_hype_score'),
        'Community': data.get('community'),
        'PageRank': data.get('pagerank'),
        'Degree': data.get('degree_centrality')
    })

df_nodes = pd.DataFrame(summary_data).sort_values(by='PageRank', ascending=False)
df_nodes

[OK] Working directory set to project root: C:\Users\RANADEEP\Documents\BS & Hype Analyzer project


Graph Nodes (Outlets): 11
Graph Edges (Echo Links): 13


,Outlet,Category,Avg Hype,Community,PageRank,Degree
2,CNBC Markets,Financial Media,0.552,0,0.291,0.8
9,Wall Street Journal Markets,Financial News,0.148,1,0.158,0.4
7,Financial Times Markets,Institutional Finance,0.142,1,0.101,0.3
3,Yahoo Finance,Retail Finance,0.126,1,0.088,0.2
5,The Verge Tech,Tech Journalism,0.234,0,0.075,0.2
0,Bloomberg Technology,Technology & Finance,0.121,0,0.072,0.2
1,Reuters Business,Macro & Markets,0.089,1,0.071,0.2
10,Crypto Alpha Moonshots,Social Media / Video,0.686,0,0.046,0.1
8,MarketWatch Bulletins,Retail Trading,0.202,0,0.044,0.1
6,CoinDesk Crypto News,Crypto & Web3,0.274,0,0.039,0.1


## 2. Interactive Force-Directed Network Graph (Plotly)

- **Node Color**: Average BS / Hype score (Viridis/Plasma).
- **Node Size**: PageRank network influence (how central the outlet is to the broader echo system).
- **Hover**: Inspect community clusters, category, and flagged rates.

In [2]:
fig_network = plot_echo_chamber_network_plotly(G)
fig_network.show()

## 3. Detailed Edge Analysis: Shared Narratives & Common Entities

Examine which outlets form the strongest narrative echo loops and which entities they co-amplify.

In [3]:
edge_records = []
for u, v, d in G.edges(data=True):
    edge_records.append({
        'Source Outlet': u,
        'Target Outlet': v,
        'Echo Strength': d.get('weight'),
        'Co-Amplified Entities': ', '.join(d.get('common_entities', []))
    })

df_edges = pd.DataFrame(edge_records).sort_values(by='Echo Strength', ascending=False)
df_edges.head(10)

,Source Outlet,Target Outlet,Echo Strength,Co-Amplified Entities
4,CNBC Markets,Yahoo Finance,0.2462,"TESLA, APPLE, S&P 500, BITCOIN, NVIDIA, WALL S..."
11,Yahoo Finance,Wall Street Journal Markets,0.2334,"APPLE, S&P 500, NVIDIA, WALL STREET, TREASURY"
9,CNBC Markets,Wall Street Journal Markets,0.2327,"NVIDIA, WALL STREET, APPLE, S&P 500"
12,Financial Times Markets,Wall Street Journal Markets,0.2309,"WALL STREET, S&P 500, TREASURY, FEDERAL RESERVE"
3,Reuters Business,Wall Street Journal Markets,0.2157,"TREASURY, FEDERAL RESERVE"
10,CNBC Markets,Crypto Alpha Moonshots,0.2005,"BITCOIN, WALL STREET"
5,CNBC Markets,The Verge Tech,0.1995,"MICROSOFT, TESLA, APPLE"
8,CNBC Markets,MarketWatch Bulletins,0.1869,"WALL STREET, S&P 500, FED"
0,Bloomberg Technology,CNBC Markets,0.1842,"MICROSOFT, WALL STREET, NVIDIA, APPLE"
7,CNBC Markets,Financial Times Markets,0.1672,"WALL STREET, S&P 500"


## 4. Export Physics-Based PyVis Simulation

Generate an interactive HTML graph using PyVis with real-time ForceAtlas2 physics.

In [4]:
html_path = export_pyvis_network_html(G, FIGURES_DIR / 'echo_chamber_graph.html')
print(f'[OK] PyVis graph exported to {html_path}')
print('You can open this HTML file directly in any browser for interactive physics manipulation!')
print('\nProceed to Notebook 04 for the live executive dashboard!')

[OK] PyVis graph exported to C:\Users\RANADEEP\Documents\BS & Hype Analyzer project\reports\figures\echo_chamber_graph.html
You can open this HTML file directly in any browser for interactive physics manipulation!

Proceed to Notebook 04 for the live executive dashboard!
